# AutoMLChain - Smoke Tests

Este notebook demonstra o workflow completo do AutoMLChain usando o MockProvider.

**Aviso**: Este notebook usa apenas o MockProvider e não faz chamadas reais a APIs externas.

In [ ]:
# Setup - Instalar dependências e importar módulos
!pip install -e . --quiet 2>/dev/null || pip install -e . -q

import warnings
warnings.filterwarnings('ignore')

from automlchain import (
    AutoMLPipeline,
    Dataset,
    EvaluationSuite,
    RMSE, F1, MAE, Accuracy,
)
from automlchain.providers import MockProvider, PricingProvider
from automlchain.prompts import PromptTemplate, PromptEngine

## 1. Dataset Management

In [ ]:
# Criar dataset de exemplo
data = [
    {"input": "Great product, highly recommend!", "output": "positive", "score": 5},
    {"input": "Terrible service, never again.", "output": "negative", "score": 1},
    {"input": "It was okay, nothing special.", "output": "neutral", "score": 3},
    {"input": "Amazing quality for the price!", "output": "positive", "score": 5},
    {"input": "Disappointing, expected better.", "output": "negative", "score": 2},
    {"input": "Average product, as expected.", "output": "neutral", "score": 3},
    {"input": "Exceeded my expectations!", "output": "positive", "score": 5},
    {"input": "Waste of money.", "output": "negative", "score": 1},
]

dataset = Dataset(data=data, format="jsonl")
print(f"Dataset criado com {len(dataset)} amostras")
print(f"\nPrimeira amostra: {dataset[0]}")

In [ ]:
# Filtrar dataset
high_score = dataset.filter(lambda x: x["score"] >= 4)
print(f"Amostras com score >= 4: {len(high_score)}")

## 2. Provider Setup

In [ ]:
# Usar MockProvider para demonstração
provider = MockProvider(
    api_key="test_key",
    simulate_duration=1.0,  # Simula 1 segundo por job
    failure_rate=0.0
)
print(f"Provider: {provider.name}")

# Verificar custos de pricing
pricing = PricingProvider("replicate")
print(f"\nCusto de training (llama-3-8b): ${pricing.get_training_cost('meta/llama-3-8b'):.4f}/1M tokens")
print(f"Custo de inference (llama-3-8b): ${pricing.get_inference_cost('meta/llama-3-8b'):.4f}/1M tokens")

## 3. Training Workflow

In [ ]:
# Criar pipeline com mock provider
pipeline = AutoMLPipeline(provider=provider)

# Upload dataset
pipeline.upload_dataset(dataset)
print("Dataset carregado no pipeline")

In [ ]:
# Iniciar training (simulado)
import time

job = pipeline.train(
    model="meta/llama-3-8b",
    epochs=2,
    batch_size=4,
)

print(f"Job iniciado: {job.job_id}")
print(f"Modelo: {job.model}")
print(f"Status inicial: {job.status}")

In [ ]:
# Aguardar completion
print("Aguardando treinamento...")
start = time.time()

while True:
    status = provider.get_job_status(job.job_id)
    print(f"  Status: {status.status}, Progress: {status.progress:.1f}%")
    if status.status in ["completed", "failed"]:
        break
    time.sleep(0.5)

elapsed = time.time() - start
print(f"\nTreinamento concluído em {elapsed:.1f}s")
print(f"Status final: {status.status}")

## 4. Evaluation

In [ ]:
# Simular predictions do modelo treinado
predictions = [
    "positive", "negative", "neutral", "positive",
    "negative", "neutral", "positive", "negative"
]

references = [d["output"] for d in data]

print("Predictions:", predictions)
print("References:", references)

In [ ]:
# Criar evaluation suite
suite = EvaluationSuite()

# Adicionar métricas customizadas
suite.add_metric("f1", F1())
suite.add_metric("accuracy", Accuracy())

# Avaliar
result = suite.evaluate(predictions, references)
print("\n=== Evaluation Results ===")
print(result)

In [ ]:
# Comparar múltiplos modelos
models_predictions = {
    "model_v1": ["positive", "negative", "neutral", "positive", "negative", "neutral", "positive", "negative"],
    "model_v2": ["positive", "negative", "neutral", "positive", "negative", "neutral", "positive", "positive"],
    "baseline": ["positive"] * 8,  # Todos positive
}

comparison = suite.compare(models_predictions, references)
print("\n=== Model Comparison ===")
for metric_name, ranking in comparison.rankings.items():
    print(f"\n{metric_name}:")
    for i, (model, score) in enumerate(ranking, 1):
        print(f"  {i}. {model}: {score:.4f}")

## 5. Regression Metrics

In [ ]:
# Testar métricas de regressão
numeric_predictions = [4.5, 1.2, 3.0, 5.0, 2.8, 3.5, 4.8, 1.5]
numeric_references = [5.0, 1.0, 3.0, 4.5, 3.0, 3.5, 5.0, 1.0]

regression_suite = EvaluationSuite()
regression_suite.add_metric("rmse", RMSE())
regression_suite.add_metric("mae", MAE())

result = regression_suite.evaluate(numeric_predictions, numeric_references)
print("=== Regression Results ===")
print(result)

## 6. Prompt Engineering

In [ ]:
# Criar template de prompt
template = PromptTemplate(
    name="sentiment_classifier",
    template="""Analyze the following text and classify as positive, negative, or neutral.

Text: {input}

Classification: {output}"""
)

print("Template:")
print(template.template)
print(f"\nVariáveis detectadas: {template.variables}")

In [ ]:
# Formatar template
formatted = template.format(
    input="This product is amazing!",
    output="positive"
)
print("Template formatado:")
print(formatted)

In [ ]:
# Gerar variantes do template
engine = PromptEngine(seed=42)
variants = engine.generate_variants(template, n=3)

print(f"Geradas {len(variants)} variantes:")
for i, variant in enumerate(variants, 1):
    print(f"\n--- Variante {i} ---")
    print(variant.template[:100] + "...")

## 7. Cancelamento de Job

In [ ]:
# Criar um novo job para cancelar
job_to_cancel = provider.train(
    model="meta/llama-3-8b",
    training_file="test.jsonl",
)
print(f"Job criado: {job_to_cancel.job_id}")

# Cancelar
provider.cancel_job(job_to_cancel.job_id)
print("Job cancelado!")

# Verificar status
status = provider.get_job_status(job_to_cancel.job_id)
print(f"Status final: {status.status}")

## 8. Estimativa de Custos

In [ ]:
# Estimar custos de training
pricing = PricingProvider("replicate")

models = [
    "meta/llama-3-8b",
    "meta/llama-3-70b",
    "mistralai/mistral-7b"
]

print("=== Estimativa de Custos ===")
print(f"\n{'Modelo':<30} {'Custo/1M tokens':<20} {'Estimativa 3 epochs'}")
print("-" * 70)

for model in models:
    cost_per_token = pricing.get_training_cost(model)
    estimated = pricing.estimate_training_cost(
        model=model,
        epochs=3,
        batch_size=4,
        dataset_size_tokens=1_000_000
    )
    print(f"{model:<30} ${cost_per_token:<19.4f} ${estimated:.4f}")

---

## Conclusão

Este notebook demonstrou:

1. **Dataset Management** - Upload, filtragem
2. **Provider Setup** - MockProvider com pricing
3. **Training Workflow** - Job creation e monitoring
4. **Evaluation** - Métricas de classificação e regressão
5. **Prompt Engineering** - Templates e variantes
6. **Job Cancellation** - Cancelamento de jobs
7. **Cost Estimation** - Estimativas de custo

Para usar com providers reais (Replicate, Together AI), basta trocar o MockProvider pelo provider apropriado e configurar as variáveis de ambiente com as API keys.